## 05.06节练习参考答案

### 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os, sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"

# 本 notebook 位于 answers/ 子目录下，src 包在其上一级目录。
# 把上层目录加入 sys.path，才能 from src.pto_layers import ...
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import time
import pypto
import torch
import torch_npu
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import math

from src.pto_layers import PyPTOLinear, PyPTOReLU, PyPTOLazyLinear

本节的解答思路参考了 [《动手学深度学习》习题解答](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/ch05/ch05) 。

原解答基于 NVIDIA GPU 与 CUDA 实现。本节沿用其解题思路，将其中与设备相关的代码改写为基于 Ascend NPU 与 torch_npu 的实现；涉及网络结构的练习（5.6.2）补充了 PyPTO 版本，其余以张量计算为主的练习则直接使用 PyTorch 原生算子在 NPU 上执行。此外，由于 NPU 上的算子以异步方式下发，计时前调用 `torch.npu.synchronize()` 以获得稳定结果。

### 练习5.6.1
尝试一个计算量更大的任务，比如大矩阵的乘法，看看 CPU 和 NPU 之间的速度差异。再试一个计算量很小的任务呢？

**解答：**  

计算量很大的任务：使用 NPU 速度明显更快

计算量很小的任务：CPU 速度可能更快，因为数据传输到 NPU 需要时间

In [3]:
# 计算量较大的任务
X = torch.rand((10000, 10000))
Y = X.npu(0)

time_start = time.time()
Z = torch.mm(X, X)
time_end = time.time()
print(f'cpu time cost: {round((time_end - time_start) * 1000, 2)}ms')

torch.npu.synchronize()
time_start = time.time()
Z = torch.mm(Y, Y)
torch.npu.synchronize()
time_end = time.time()
print(f'npu time cost: {round((time_end - time_start) * 1000, 2)}ms')

# 计算量很小的任务
X = torch.rand((100, 100))
Y = X.npu(0)

time_start = time.time()
Z = torch.mm(X, X)
time_end = time.time()
print(f'cpu time cost: {round((time_end - time_start) * 1000)}ms')

torch.npu.synchronize()
time_start = time.time()
Z = torch.mm(Y, Y)
torch.npu.synchronize()
time_end = time.time()
print(f'npu time cost: {round((time_end - time_start) * 1000)}ms')

cpu time cost: 1299.55ms
npu time cost: 152.56ms
cpu time cost: 0ms
npu time cost: 46ms


### 练习5.6.2
我们应该如何在 NPU 上读写模型参数？

**解答：**
使用 `net.to(device)` 将模型迁移到 NPU 上，然后再按照之前的方法读写参数。

In [4]:
class MLP(nn.Module):             # 定义 MLP 类
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)   # 定义隐藏层，输入尺寸为 20，输出尺寸为 256
        self.output = nn.Linear(256, 10)   # 定义输出层，输入尺寸为 256，输出尺寸为 10

    def forward(self, x):          # 定义前向传播函数
        return self.output(F.relu(self.hidden(x)))  # 使用 ReLU 激活函数，计算隐藏层和输出层的输出

# 选择 NPU，没有 NPU 就选 CPU
device = torch.device("npu:0" if torch.npu.is_available() else "cpu")
# 创建模型实例对象
net = MLP()
# 将模型参数传输到 NPU 上
net.to(device)
# 访问模型参数（只打印参数名、形状和设备，便于阅读）
for name, param in net.state_dict().items():
    print(f"{name}: shape={tuple(param.shape)}, device={param.device}")

hidden.weight: shape=(256, 20), device=npu:0
hidden.bias: shape=(256,), device=npu:0
output.weight: shape=(10, 256), device=npu:0
output.bias: shape=(10,), device=npu:0


**PyPTO 版**

In [5]:
class MLP(nn.Module):             # 定义 MLP 类
    def __init__(self):
        super().__init__()
        self.hidden = PyPTOLinear(20, 256)   # 定义隐藏层，输入尺寸为 20，输出尺寸为 256
        self.output = PyPTOLinear(256, 10)   # 定义输出层，输入尺寸为 256，输出尺寸为 10

    def forward(self, x):          # 定义前向传播函数
        return self.output(F.relu(self.hidden(x)))  # 使用 ReLU 激活函数，计算隐藏层和输出层的输出

# 创建模型实例对象（PyPTOLinear 默认将参数创建在 npu:0 上）
net = MLP()
# 访问模型参数（只打印参数名、形状和设备，便于阅读）
for name, param in net.state_dict().items():
    print(f"{name}: shape={tuple(param.shape)}, device={param.device}")

hidden.weight: shape=(256, 20), device=npu:0
hidden.bias: shape=(256,), device=npu:0
output.weight: shape=(10, 256), device=npu:0
output.bias: shape=(10,), device=npu:0


### 练习5.6.3
测量计算1000个 100×100 矩阵的矩阵乘法所需的时间，并记录输出矩阵的 Frobenius 范数，一次记录一个结果，而不是在 NPU 上保存日志并仅传输最终结果。

**解答：**
中文版翻译有点问题，英文原版这句话是：

> Measure the time it takes to compute 1000 matrix-matrix multiplications of $100 \times 100$ matrices and log the Frobenius norm of the output matrix one result at a time vs. keeping a log on the GPU and transferring only the final result.

所以这道题的本质还是希望我们做个比较。

实验一：仅记录1000次$100 \times 100$矩阵相乘所用的时间，范数结果保留在 NPU 上，不逐次搬运到 CPU。

实验二：记录1000次$100 \times 100$矩阵相乘所用的时间，并每次将范数打印输出（每次打印都会触发 NPU→CPU 的数据搬运）。

In [6]:
device = torch.device("npu:0" if torch.npu.is_available() else "cpu")

# 生成随机矩阵并放到 device 上
matrices = [torch.randn(100, 100, device=device) for i in range(1000)]

# 实验一：仅计算 Frobenius 范数，结果保留在 NPU 上，不逐次搬运到 CPU
start_time_1 = time.time()
for i in range(1000):
    result = torch.mm(matrices[i], matrices[i].t())
    frobenius_norm = torch.norm(result)
torch.npu.synchronize()
end_time_1 = time.time()
print(f"实验一 Time taken: {end_time_1 - start_time_1:.4f}s")

# 实验二：计算范数并逐次搬运到 CPU（每次 .item() 都会触发 NPU→CPU 的数据搬运）
norms = []
start_time_2 = time.time()
for i in range(1000):
    result = torch.mm(matrices[i], matrices[i].t())
    frobenius_norm = torch.norm(result)
    norms.append(frobenius_norm.item())
end_time_2 = time.time()
print(f"实验二 Time taken: {end_time_2 - start_time_2:.4f}s")

# 仅打印前 5 个和后 5 个范数结果作为示例
print(f"前 5 个范数：{[round(n, 4) for n in norms[:5]]}")
print(f"后 5 个范数：{[round(n, 4) for n in norms[-5:]]}")
print(f"实验一消耗时间：{end_time_1 - start_time_1:.4f}s，实验二消耗时间：{end_time_2 - start_time_2:.4f}s")

实验一 Time taken: 0.0741s
实验二 Time taken: 0.0860s
前 5 个范数：[1425.7274, 1424.7217, 1388.9281, 1387.2322, 1398.4214]
后 5 个范数：[1428.724, 1416.0704, 1435.4037, 1414.0779, 1424.2792]
实验一消耗时间：0.0741s，实验二消耗时间：0.0860s


### 练习5.6.4
测量同时在两个 NPU 上执行两个矩阵乘法与在一个 NPU 上按顺序执行两个矩阵乘法所需的时间。提示：应该看到近乎线性的缩放。

**解答：**

将两个矩阵乘法分别放到两张 NPU 上并行执行，通常会比在单张 NPU 上顺序执行更快，但实际加速比取决于矩阵大小、硬件配置和算子实现。

受限于单卡环境，下面仅给出在单张 NPU 上顺序执行两次矩阵乘法的实验。若有两张 NPU，可将两次乘法分别放到 `npu:0` 与 `npu:1` 上，并借助各自的计算流实现并行。

In [7]:
# 创建两个随机矩阵并放到 NPU 上
a = torch.randn(10000, 10000, device='npu')
b = torch.randn(10000, 10000, device='npu')

# 单 NPU 上顺序执行两次矩阵乘法
torch.npu.synchronize()
start_time = time.time()
c1 = torch.matmul(a, b)
c2 = torch.matmul(a, b)
torch.npu.synchronize()
end_time = time.time()
sequential_time = end_time - start_time

print(f"Sequential time: {sequential_time:.8f} seconds")

Sequential time: 0.09191298 seconds
